In [21]:
import pandas as pd
import requests
import re
import time

from bs4 import BeautifulSoup
from urllib.parse import quote_plus
from pathlib import Path

In [22]:
df = pd.read_excel(r"D:\Proyek FEB\Publikasi internasional.xlsx")
df.head()

,No,Title,Authors,Journal,SCOPUS Q,Tahun,Volume / Issue,LINK DOI/ARTIKEL,SCOPUS SITASI
0,1,Ethnic identity and internal migration decisio...,ILMIAWAN AUWALIN,Journal of Ethnic and Migration Studies,?,2019-01-11 00:00:00,"Vol. 14, Issue 13",10.1080/1369183X.2018.1561252,?
1,2,The role of service quality within Indonesian ...,BADRI MUNIR SUKOCO,Journal of Islamic Marketing,?,2020-01-14 00:00:00,"Vol. 11, Issue 1",10.1108/JIMA-03-2017-0033,?
2,3,Developing Islamic crowdfunding website platfo...,MUHAMAD NAFIK HADI RYANDONO,Journal of Islamic Marketing,?,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,?
3,4,Developing Islamic crowdfunding website platfo...,ACHSANIA HENDRATMI,Journal of Islamic Marketing,?,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,?
4,5,Developing Islamic crowdfunding website platfo...,PUJI SUCIA SUKMANINGRUM,Journal of Islamic Marketing,?,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,?


In [23]:
COL_TITLE = "Title"
COL_Q = "SCOPUS Q"
COL_CITATION = "SCOPUS SITASI"

In [24]:
df.isnull().sum()

No                   0
Title                0
Authors              0
Journal              9
SCOPUS Q             0
Tahun                2
Volume / Issue      96
LINK DOI/ARTIKEL    54
SCOPUS SITASI        0
dtype: int64

In [25]:
COL_TITLE = "Title"
COL_Q = "SCOPUS Q"
COL_CITATION = "SCOPUS SITASI"
COL_LINK = "LINK DOI/ARTIKEL"

In [26]:
def is_empty_value(value):
    if pd.isna(value):
        return True
    
    text = str(value).strip()
    
    if text == "":
        return True
    
    if text.lower() in [
        "nan",
        "none",
        "null",
        "-",
        "?",
        "??",
        "???",
        "n/a",
        "na",
        "tidak ada",
        "not found"
    ]:
        return True
    
    return False

In [27]:
def parse_quartile(text):
    if not text:
        return None
    
    text = str(text).strip()
    
    if re.search(r"\bno-?Q\b", text, flags=re.IGNORECASE):
        return "no-Q"
    
    match = re.search(r"\bQ\s*([1-4])\b", text, flags=re.IGNORECASE)
    
    if match:
        return f"Q{match.group(1)}"
    
    return None


def parse_citation(text):
    if not text:
        return None
    
    text = str(text).strip()
    
    match = re.search(r"(\d+)", text)
    
    if match:
        return int(match.group(1))
    
    return None

In [28]:
def scrape_sinta_q_citation_by_title(title, sleep_time=1.5):
    base_url = "https://sinta.kemdiktisaintek.go.id/scopus/"
    search_url = f"{base_url}?q={quote_plus(str(title))}"
    
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    
    try:
        response = requests.get(search_url, headers=headers, timeout=30)
        time.sleep(sleep_time)
        
        if response.status_code != 200:
            return {
                "input_title": title,
                "matched_title": None,
                "sinta_year": None,
                "scopus_q": None,
                "scopus_citation": None,
                "citation_text": None,
                "quartile_text": None,
                "status": "request_failed",
                "source_url": search_url,
                "http_status": response.status_code
            }
        
        soup = BeautifulSoup(response.text, "html.parser")
        
        first_item = soup.select_one("div.ar-list-item")
        
        if first_item is None:
            return {
                "input_title": title,
                "matched_title": None,
                "sinta_year": None,
                "scopus_q": None,
                "scopus_citation": None,
                "citation_text": None,
                "quartile_text": None,
                "status": "not_found",
                "source_url": search_url,
                "http_status": response.status_code
            }
        
        title_el = first_item.select_one(".ar-title")
        year_el = first_item.select_one(".ar-year")
        cited_el = first_item.select_one(".ar-cited")
        quartile_el = first_item.select_one(".ar-quartile")
        
        matched_title = title_el.get_text(" ", strip=True) if title_el else None
        sinta_year = year_el.get_text(" ", strip=True) if year_el else None
        
        citation_text = cited_el.get_text(" ", strip=True) if cited_el else None
        quartile_text = quartile_el.get_text(" ", strip=True) if quartile_el else None
        
        scopus_q = parse_quartile(quartile_text)
        scopus_citation = parse_citation(citation_text)
        
        return {
            "input_title": title,
            "matched_title": matched_title,
            "sinta_year": sinta_year,
            "scopus_q": scopus_q,
            "scopus_citation": scopus_citation,
            "citation_text": citation_text,
            "quartile_text": quartile_text,
            "status": "found",
            "source_url": search_url,
            "http_status": response.status_code
        }
    
    except Exception as e:
        return {
            "input_title": title,
            "matched_title": None,
            "sinta_year": None,
            "scopus_q": None,
            "scopus_citation": None,
            "citation_text": None,
            "quartile_text": None,
            "status": "error",
            "source_url": search_url,
            "error": str(e)
        }

In [29]:
def row_needs_q_citation_scraping(row):
    q_empty = is_empty_value(row.get(COL_Q))
    citation_empty = is_empty_value(row.get(COL_CITATION))
    
    return q_empty or citation_empty

In [30]:
rows_need_scraping = df[df.apply(row_needs_q_citation_scraping, axis=1)].copy()

unique_titles_to_scrape = (
    rows_need_scraping[COL_TITLE]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Jumlah judul unik yang perlu scraping:", len(unique_titles_to_scrape))

Jumlah judul unik yang perlu scraping: 1332


In [ ]:
results = []

for i, title in enumerate(unique_titles_to_scrape, start=1):
    print(f"{i}/{len(unique_titles_to_scrape)} - {str(title)[:90]}")
    
    result = scrape_sinta_q_citation_by_title(title)
    results.append(result)
    
    if i % 50 == 0:
        pd.DataFrame(results).to_excel("sinta_q_citation_checkpoint.xlsx", index=False)

sinta_results_df = pd.DataFrame(results)
sinta_results_df.to_excel("sinta_q_citation_results.xlsx", index=False)

sinta_results_df.head()

1/1332 - Ethnic identity and internal migration decision in Indonesia
2/1332 - The role of service quality within Indonesian customers satisfaction and loyalty and its i
3/1332 - Developing Islamic crowdfunding website platform for startup companies in Indonesia
4/1332 - Exploration of pilgrimage tourism in Indonesia
5/1332 - A critical assessment of retail sovereign sukuk in Indonesia
6/1332 - The impact of auditors’ awareness of the profession’s reputation for independence on audit
7/1332 - Exploring the role of visual aesthetics and presentation modality in luxury fashion brand 
8/1332 - Anger punishes, compassion forgives: How discrete emotions mitigate double standards in co
9/1332 - Managing paradoxes of innovation in an Indonesian TV group
10/1332 - Competitiveness and cost behaviour: evidence from the retail industry
11/1332 - Effects of Halal social media and customer engagement on brand satisfaction of Muslim cust
12/1332 - Islamic microfinance institution: Survey data from I

In [ ]:
sinta_results_df["status"].value_counts(dropna=False)

In [ ]:
sinta_results_df["scopus_citation"].notna().sum()

In [ ]:
error_titles = (
    sinta_results_df[sinta_results_df["status"] == "error"]["input_title"]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Jumlah error yang akan diulang:", len(error_titles))

In [ ]:
retry_results = []

for i, title in enumerate(error_titles, start=1):
    print(f"Retry {i}/{len(error_titles)} - {str(title)[:90]}")
    
    result = scrape_sinta_q_citation_by_title(title, sleep_time=2.5)
    retry_results.append(result)
    
    if i % 25 == 0:
        pd.DataFrame(retry_results).to_excel("sinta_q_citation_retry_checkpoint.xlsx", index=False)

retry_df = pd.DataFrame(retry_results)
retry_df.to_excel("sinta_q_citation_retry_results.xlsx", index=False)

retry_df["status"].value_counts(dropna=False)

In [ ]:
def normalize_title(value):
    if pd.isna(value):
        return ""
    
    text = str(value).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.strip()

In [ ]:
if len(error_titles) > 0:
    sinta_results_df["title_key"] = sinta_results_df["input_title"].apply(normalize_title)
    retry_df["title_key"] = retry_df["input_title"].apply(normalize_title)
    
    old_non_error = sinta_results_df[sinta_results_df["status"] != "error"].copy()
    
    sinta_results_fixed = pd.concat(
        [old_non_error, retry_df],
        ignore_index=True
    )
    
    sinta_results_fixed = (
        sinta_results_fixed
        .drop_duplicates(subset=["title_key"], keep="last")
        .reset_index(drop=True)
    )
else:
    sinta_results_fixed = sinta_results_df.copy()

In [ ]:
sinta_results_fixed["status"].value_counts(dropna=False)

In [ ]:
sinta_results_fixed.to_excel("sinta_q_citation_results_fixed.xlsx", index=False)